In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier



In [35]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [40]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [41]:
train.shape

(8693, 14)

In [42]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [43]:
train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [70]:
y = train["Transported"]
train = train.drop("Transported", axis=1)


In [ ]:
def clean_data(df):
    df = df.copy()
    

    df = df.drop(columns=["PassengerId", "Name", "Cabin"], errors='ignore')
    
    num_cols = ["Age", "RoomService", "FoodCourt", 
                "ShoppingMall", "Spa", "VRDeck"]
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    

    cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    
    return df


In [ ]:
def feature_engineering(df):
    df = df.copy()
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    return df



In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_features(train_df, test_df):
    
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
    
    for col in cat_cols:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col])
        test_df[col] = le.transform(test_df[col])
    
    return train_df, test_df


In [74]:
from sklearn.preprocessing import StandardScaler

def scale_features(train_df, test_df):
    """
    Standard scale numerical features
    """
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    scale_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck", "TotalSpend"]
    
    scaler = StandardScaler()
    train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
    test_df[scale_cols] = scaler.transform(test_df[scale_cols])
    
    return train_df, test_df


In [75]:
def preprocess_pipeline(train_df, test_df):
    # Feature Engineering
    train_df = feature_engineering(train_df)
    test_df = feature_engineering(test_df)
    
    # Encoding
    train_df, test_df = encode_features(train_df, test_df)
    
    # Scaling
    train_df, test_df = scale_features(train_df, test_df)
    
    # Align columns
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)
    
    return train_df, test_df


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

def train_model(X, y):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    val_preds = model.predict(X_val)
    print("Validation Accuracy:", accuracy_score(y_val, val_preds))
    print(classification_report(y_val, val_preds))
    
    return model


In [ ]:
def predict_and_submit(model, test_df, submission_file="sample_submission.csv"):

    if "Transported" in test_df.columns:
        test_df = test_df.drop("Transported", axis=1)
    
    preds = model.predict(test_df)
    
    submission = pd.read_csv(submission_file)
    submission["Transported"] = preds
    submission.to_csv("submission.csv", index=False)
    
    print("Submission file ready ✅")


In [80]:
# Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Separate target
y = train["Transported"]
train = train.drop("Transported", axis=1)

# 1. Data Cleaning
train_clean = clean_data(train)
test_clean = clean_data(test)

# 2. Preprocessing
X, X_test = preprocess_pipeline(train_clean, test_clean)

# 3. Train Model
model = train_model(X, y)

# 4. Predict & Submission
predict_and_submit(model, X_test, submission_file="sample_submission.csv")


C:\Users\gold.lab10\AppData\Local\Temp\ipykernel_1568\735756303.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
C:\Users\gold.lab10\AppData\Local\Temp\ipykernel_1568\735756303.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


Validation Accuracy: 0.7751581368602645
              precision    recall  f1-score   support

       False       0.78      0.76      0.77       861
        True       0.77      0.79      0.78       878

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739

Submission file ready ✅
